# XAI Method Selection for the Bristol Regional Food Network AI System

This notebook records the explainability method selected for each AI task in the project. The goal is to support transparency, trust, and administrator visibility by making model decisions easier to understand.

## Why explainability is needed

The case study requires the system to provide transparency in predictions and recommendations. This is important because:

- producers need to understand why produce was graded as high or low quality,
- customers and administrators should understand why recommendations are shown,
- the system should support fairness, accountability, and trust.

For this reason, we compared three explainable AI methods: Grad-CAM, SHAP DeepExplainer, and LIME.

## Method comparison

### 1. Grad-CAM
Grad-CAM (Gradient-weighted Class Activation Mapping) is an explainability method designed for convolutional neural networks used in computer vision. It produces a heatmap on top of the input image to highlight the image regions that contributed most to the model's prediction.

**Strengths**
- Very suitable for image classification tasks.
- Gives visual explanations that are easy for humans to interpret.
- Helps show which parts of a fruit or vegetable image influenced the healthy/rotten prediction.

**Weaknesses**
- Mainly suited to CNN-style vision models.
- Does not provide the same style of feature-level explanation as tabular methods.

### 2. SHAP DeepExplainer
SHAP DeepExplainer estimates the contribution of each input feature to the model prediction. It is useful for structured or tabular data and can explain which features push a prediction higher or lower.

**Strengths**
- Well suited to feature-based models and structured inputs.
- Gives both local explanations for one prediction and broader insights into feature importance.
- Useful for explaining recommendation systems and reorder decisions.

**Weaknesses**
- Less intuitive than image heatmaps for computer vision tasks.
- Can be more computationally expensive depending on the model and dataset.

### 3. LIME
LIME (Local Interpretable Model-agnostic Explanations) explains an individual prediction by approximating the model locally with a simpler interpretable model.

**Strengths**
- Model-agnostic, so it can be used with many different models.
- Useful for local explanations of single predictions.

**Weaknesses**
- Explanations may vary between runs and can be less stable.
- Usually slower and less natural than Grad-CAM for image-based CNN tasks.
- For this project, it is less direct than SHAP for Task 1 and less visual than Grad-CAM for Task 2.

## Selected XAI methods

### Task 2: Computer Vision Quality Model → Grad-CAM
Grad-CAM is the best choice for Task 2 because the model is a computer vision classifier working on fruit and vegetable images. Producers and administrators need to see which image regions influenced the model decision. A heatmap is the clearest way to show why an item may have been classified as healthy or rotten.

### Task 1: Purchase Prediction / Reorder Recommendation → SHAP
SHAP is the best choice for Task 1 because the recommendation system is based on structured information such as customer purchase history, order frequency, recency, and seasonal behaviour. SHAP can explain how these features contributed to a recommendation, making it more suitable than Grad-CAM or LIME for this task.

## Final decision

The chosen XAI methods for the project are:

- **Task 2 (CV quality grading): Grad-CAM**
- **Task 1 (purchase prediction / reorder recommendation): SHAP**

This combination is the best fit because it matches the type of data and model used in each task:
- image-based CNN predictions are best explained visually with Grad-CAM,
- structured recommendation predictions are best explained with feature attribution using SHAP.

LIME was considered, but it was not selected because it is less stable and less specialised for the two main model types used in this project.

---
## AA-46: Grad-CAM — Visualising Quality Grade Decisions

Grad-CAM registers forward and backward hooks on the last convolutional layer of the quality CNN. When the model makes a prediction, we trigger a backward pass on the predicted class score, pool the gradients spatially, and weight the activation maps. The resulting heatmap (resized to input size) shows which image regions most influenced the grade decision.

Below we show the Grad-CAM overlay on **5 sample images**.

In [ ]:
import sys, os
from pathlib import Path

# Ensure repo root is on sys.path
REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

print(f'Repo root: {REPO_ROOT}')

In [ ]:
import torch
from torchvision import transforms

from task2_3_4_cv_quality.xai.gradcam import GradCAM, get_target_layer

_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

DATA_DIR = REPO_ROOT / 'task2_3_4_cv_quality' / 'data' / 'raw' / 'Fruit And Vegetable Diseases Dataset'
MODEL_DIR = REPO_ROOT / 'task2_3_4_cv_quality' / 'models'

# Collect up to 5 sample images from the dataset
sample_paths = []
if DATA_DIR.exists():
    for cls_dir in sorted(DATA_DIR.iterdir()):
        if cls_dir.is_dir():
            imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
            if imgs:
                sample_paths.append(imgs[0])
        if len(sample_paths) == 5:
            break

print(f'Found {len(sample_paths)} sample images')
for p in sample_paths:
    print(' ', p.parent.name, '/', p.name)

In [ ]:
# Load model (skip if not yet trained)
ckpt_candidates = list(MODEL_DIR.glob('*.pth')) + list(MODEL_DIR.glob('*.pt'))
model_loaded = False
gradcam = None

if ckpt_candidates and sample_paths:
    try:
        from task2_3_4_cv_quality.src.model import get_model
        from task2_3_4_cv_quality.src.train import create_dataloaders

        bundle = create_dataloaders(verbose=False)
        model = get_model(num_classes=len(bundle.classes), model_type='resnet50', pretrained=False)
        ckpt = torch.load(ckpt_candidates[0], map_location='cpu')
        state = ckpt.get('model_state_dict', ckpt)
        model.load_state_dict(state)
        model.eval()

        target_layer = get_target_layer(model, 'resnet50')
        gradcam = GradCAM(model, target_layer)
        model_loaded = True
        print('Model loaded — generating real Grad-CAM heatmaps')
    except Exception as e:
        print(f'Model load failed: {e}')
        print('Falling back to synthetic demonstration')
else:
    print('No trained model or sample images found — using synthetic demonstration')

# Generate and display Grad-CAM for 5 samples (or synthetic fallback)
fig, axes = plt.subplots(5, 3, figsize=(12, 20))
fig.suptitle('Grad-CAM: Quality Grade Decision Visualisation (AA-46)', fontsize=14, fontweight='bold')

rng = np.random.default_rng(42)

produce_examples = [
    ('Apple__Healthy',    0.95, '#27ae60'),
    ('Banana__Rotten',    0.88, '#e74c3c'),
    ('Tomato__Healthy',   0.76, '#27ae60'),
    ('Strawberry__Rotten',0.91, '#e74c3c'),
    ('Orange__Healthy',   0.82, '#f39c12'),
]

for i, (cls_name, conf, grade_color) in enumerate(produce_examples):
    if model_loaded and i < len(sample_paths):
        img = Image.open(sample_paths[i]).convert('RGB').resize((224, 224))
        tensor = _transform(img).unsqueeze(0)
        cam, pred_idx = gradcam.generate(tensor)
        overlay = gradcam.overlay_on_image(img, cam)
        img_np = np.array(img)
        predicted_label = f'Class {pred_idx}'
    else:
        # Synthetic demonstration image
        img_np = rng.integers(80, 200, (224, 224, 3), dtype=np.uint8)
        # Synthetic cam: blob in centre or corner depending on healthy/rotten
        cam = np.zeros((224, 224), dtype=np.float32)
        if 'Healthy' in cls_name:
            cy, cx = 112, 112
        else:
            cy, cx = rng.integers(50, 174, 2)
        yy, xx = np.ogrid[:224, :224]
        cam = np.exp(-((yy - cy)**2 + (xx - cx)**2) / (2 * 40**2)).astype(np.float32)
        import cv2
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = np.clip(0.6*img_np + 0.4*heatmap, 0, 255).astype(np.uint8)
        predicted_label = cls_name

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title('Original Image', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[i, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f'Overlay\n{predicted_label} ({conf:.0%})', fontsize=9, color=grade_color)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()
print('\nGrad-CAM heatmaps highlight the image regions that drove each quality grade decision.')

---
## AA-47: SHAP — Explaining Quality Score Derivation

`SHAPExplainer` wraps `shap.GradientExplainer` to attribute each pixel's contribution to the quality score outputs. Below we show:
1. The SHAP pixel-attribution map for a sample image.
2. A human-readable `explain_grade()` description for each grade.

In [ ]:
from task2_3_4_cv_quality.xai.shap_explainer import SHAPExplainer
from task2_3_4_cv_quality.src.grading import grade_produce

# --- Grade explanations (always runnable) ---
grade_examples = [
    ('Apple__Healthy',    0.95),
    ('Banana__Healthy',   0.72),
    ('Tomato__Rotten',    0.85),
]

print('=== Grade Explanations (AA-47) ===')
print()
explainer_instance = SHAPExplainer.__new__(SHAPExplainer)  # dummy — no model needed for explain_grade

for cls_name, conf in grade_examples:
    result = grade_produce({'predicted_class': cls_name, 'confidence': conf})
    explanation = SHAPExplainer.explain_grade(
        explainer_instance,
        result['grade'],
        {'color_score': result['color_score'],
         'size_score':  result['size_score'],
         'ripeness_score': result['ripeness_score']},
    )
    print(f'Input : {cls_name}  confidence={conf:.0%}')
    print(f'Output: {explanation}')
    print()

In [ ]:
# SHAP pixel attribution (requires trained model + shap package)
try:
    import shap
    _shap_available = True
except ImportError:
    _shap_available = False
    print('shap not installed. Install with: pip install shap')
    print('Showing synthetic attribution demonstration instead.')

if model_loaded and _shap_available and sample_paths:
    # Use first 5 images as background, first image as query
    bg_tensors = torch.stack([_transform(Image.open(p).convert('RGB')) for p in sample_paths[:5]])
    shap_exp = SHAPExplainer(model, bg_tensors)
    query = _transform(Image.open(sample_paths[0]).convert('RGB')).unsqueeze(0)
    shap_vals = shap_exp.explain_prediction(query)
    # Show first 4 classes
    shap_exp.plot_shap(shap_vals[:4], query.numpy(), max_display=4)
else:
    # Synthetic attribution visualisation
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    rng2 = np.random.default_rng(7)
    synth_img = rng2.integers(80, 200, (224, 224, 3), dtype=np.uint8)
    
    # Positive SHAP: centre blob (model pays attention here)
    pos_map = np.zeros((224, 224))
    yy, xx = np.ogrid[:224, :224]
    pos_map = np.exp(-((yy-112)**2+(xx-112)**2)/(2*45**2)) * 0.8
    neg_map = np.exp(-((yy-30)**2+(xx-30)**2)/(2*25**2)) * -0.3
    combined = pos_map + neg_map

    axes[0].imshow(synth_img)
    axes[0].set_title('Input Image (synthetic demo)', fontsize=10)
    axes[0].axis('off')

    im = axes[1].imshow(combined, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    plt.colorbar(im, ax=axes[1], fraction=0.04)
    axes[1].set_title('SHAP Pixel Attribution\n(red=positive, blue=negative)', fontsize=10)
    axes[1].axis('off')

    plt.suptitle('SHAP Quality Score Attribution (AA-47)', fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\nSHAP attribution shows which pixel regions drove the quality score prediction.')

---
## AA-48: SHAP — Explaining Reorder Recommendations (Task 1)

`explain_reorder()` uses `shap.TreeExplainer` (compatible with RandomForest and XGBoost) to find the top 3 features that drove a reorder prediction. `explain_forecast()` returns a plain-language seasonal explanation.

In [ ]:
from task2_3_4_cv_quality.xai.shap_explainer import explain_reorder, explain_forecast, plot_shap_bar
from pathlib import Path
import joblib

T1_MODEL = REPO_ROOT / 'task1_purchase_prediction' / 'models' / 'reorder_model.pkl'
T1_ORDERS = REPO_ROOT / 'task1_purchase_prediction' / 'data' / 'raw' / 'orders.csv'

reorder_explanation = None

if T1_MODEL.exists() and T1_ORDERS.exists():
    try:
        import pandas as pd
        from task1_purchase_prediction.src.preprocess import load_orders, load_class_names
        from task1_purchase_prediction.src.evaluate import build_training_dataset

        model_rf = joblib.load(T1_MODEL)
        orders_df = load_orders(T1_ORDERS)
        class_names = load_class_names(REPO_ROOT / 'task1_purchase_prediction' / 'data' / 'raw' / 'products.csv')
        X, y = build_training_dataset(orders_df, class_names)

        # Use a single row as example
        X_instance = X.iloc[[0]]
        reorder_explanation = explain_reorder(model_rf, X_instance)

        print('=== Reorder SHAP Explanation (AA-48) ===')
        print(f'Predicted class : {reorder_explanation["predicted_class"]}')
        print(f'Summary         : {reorder_explanation["summary"]}')
        print()
        print('Top 3 feature contributions:')
        for feat in reorder_explanation['top_features']:
            print(f'  {feat["description"]}')

        fig, ax = plt.subplots(figsize=(9, 3))
        plot_shap_bar(reorder_explanation, title='SHAP Feature Importance — Reorder Prediction (AA-48)', ax=ax)
        plt.show()
    except Exception as e:
        print(f'SHAP reorder explanation failed: {e}')
        print('Model may not be trained yet. Run task1_purchase_prediction/src/model.py first.')
else:
    print('Task 1 model or orders data not found — showing synthetic demonstration.')

    import pandas as pd
    # Synthetic demonstration
    products = ['Apples', 'Bananas', 'Tomatoes', 'Oranges', 'Strawberries']
    synth_explanation = {
        'predicted_class': 'Tomatoes',
        'summary': 'The main driver was purchase_frequency of \'Tomatoes\' (SHAP=+0.312).',
        'top_features': [
            {'feature': 'freq_Tomatoes',    'shap_value':  0.312, 'description': 'purchase_frequency of Tomatoes increased probability (SHAP=+0.3120)'},
            {'feature': 'qty_Tomatoes',     'shap_value':  0.187, 'description': 'total_quantity of Tomatoes increased probability (SHAP=+0.1870)'},
            {'feature': 'freq_Strawberries','shap_value': -0.094, 'description': 'purchase_frequency of Strawberries decreased probability (SHAP=-0.0940)'},
        ],
    }
    print('=== Reorder SHAP Explanation — Synthetic Demo (AA-48) ===')
    print(f'Predicted class : {synth_explanation["predicted_class"]}')
    print(f'Summary         : {synth_explanation["summary"]}')
    for feat in synth_explanation['top_features']:
        print(f'  {feat["description"]}')

    fig, ax = plt.subplots(figsize=(9, 3))
    plot_shap_bar(synth_explanation, title='SHAP Feature Importance — Reorder Prediction (AA-48)', ax=ax)
    plt.show()

In [ ]:
print('=== Seasonal Forecast Explanations (AA-48) ===')
for product, month in [('tomatoes', 7), ('strawberries', 6), ('apples', 10), ('oranges', 1)]:
    explanation = explain_forecast(product, month)
    print(explanation)
    print()

---
## AA-50: Producer Bias Audit

For each producer we compute the recommendation rate and flag producers whose rate is more than **1.5 standard deviations above** the cross-producer mean. A bar chart visualises recommendation rates per producer.

### FAT Conclusion
Fairness, Accountability and Trust (FAT) requires the system to treat all producers equally. If any producer is flagged here, it indicates their products appear disproportionately in recommendations — either because the model has learned a spurious correlation or because those products are genuinely purchased more often. Flagged producers should be investigated before deploying the model at scale.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

T1_MODEL = REPO_ROOT / 'task1_purchase_prediction' / 'models' / 'reorder_model.pkl'

audit_result = None

if T1_MODEL.exists():
    try:
        from task1_purchase_prediction.src.evaluate import bias_audit, build_predictions_df, print_bias_audit

        predictions_df = build_predictions_df()
        audit_result = bias_audit(predictions_df)
        print_bias_audit(audit_result)
    except Exception as e:
        print(f'Bias audit failed: {e}')

if audit_result is None:
    # Synthetic demonstration
    print('Using synthetic producer bias data for demonstration.')
    rng3 = np.random.default_rng(99)
    producers = [f'PROD{i:03d}' for i in range(1, 9)]
    counts = rng3.integers(30, 200, len(producers))
    total = counts.sum()
    rec_rates = counts / total
    mean_rate = rec_rates.mean()
    std_rate = rec_rates.std()
    threshold = mean_rate + 1.5 * std_rate
    flagged = rec_rates > threshold
    audit_result = {
        'per_producer': pd.DataFrame({
            'producer_id': producers,
            'sample_count': counts,
            'precision': rng3.uniform(0.55, 0.90, len(producers)).round(4),
            'recommendation_rate': rec_rates.round(4),
            'avg_confidence': rng3.uniform(0.60, 0.85, len(producers)).round(4),
            'deviation': ((rec_rates - mean_rate) / std_rate).round(3),
            'flagged': flagged,
        }),
        'mean_recommendation_rate': float(mean_rate),
        'std_recommendation_rate': float(std_rate),
        'flag_threshold_std': float(threshold),
        'flagged_producers': [p for p, f in zip(producers, flagged) if f],
    }

# Bar chart
pp = audit_result['per_producer'].sort_values('recommendation_rate', ascending=False)
colors = ['#e74c3c' if f else '#3498db' for f in pp['flagged']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(pp['producer_id'], pp['recommendation_rate'] * 100, color=colors)
ax.axhline(audit_result['mean_recommendation_rate'] * 100, color='navy',
           linestyle='--', linewidth=1.5, label='Mean')
ax.axhline(audit_result['flag_threshold_std'] * 100, color='red',
           linestyle=':', linewidth=1.5, label=f'Flag threshold (+1.5σ)')

ax.set_xlabel('Producer ID')
ax.set_ylabel('Recommendation Rate (%)')
ax.set_title('Recommendation Rate per Producer — Bias Audit (AA-50)', fontweight='bold')
flagged_patch = mpatches.Patch(color='#e74c3c', label='Flagged (>1.5σ above mean)')
normal_patch  = mpatches.Patch(color='#3498db', label='Within threshold')
ax.legend(handles=[flagged_patch, normal_patch,
                   plt.Line2D([0],[0], color='navy', linestyle='--', label='Mean'),
                   plt.Line2D([0],[0], color='red', linestyle=':', label='Flag threshold')])
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(f"\nFlagged producers: {audit_result['flagged_producers'] or 'None'}")
print(f"Mean recommendation rate : {audit_result['mean_recommendation_rate']:.4f}")
print(f"Std deviation            : {audit_result['std_recommendation_rate']:.4f}")
print(f"Flag threshold (+1.5σ)   : {audit_result['flag_threshold_std']:.4f}")

### FAT Summary

The bias audit above checks whether any producer is over-represented in model recommendations. Producers flagged at the **1.5σ threshold** should be reviewed:

- If flagged due to genuine purchasing patterns → the model is behaving correctly; no action needed.
- If flagged due to data imbalance or feature leakage → retrain with balanced sampling or add a fairness constraint.

This audit directly supports the Fairness, Accountability and Trust (FAT) requirement of the Bristol Regional Food Network case study.

---
## AA-52: Seasonal Demand Forecast Charts

`forecast_demand()` returns structured monthly data including `predicted_demand`, `last_year_demand`, and a `trend_label`. We plot a line chart for **3 products**.

In [ ]:
from task1_purchase_prediction.src.predict import forecast_demand
import matplotlib.pyplot as plt
import numpy as np

T1_ORDERS = REPO_ROOT / 'task1_purchase_prediction' / 'data' / 'raw' / 'orders.csv'
PRODUCTS_CSV = REPO_ROOT / 'task1_purchase_prediction' / 'data' / 'raw' / 'products.csv'

# Detect available products
forecast_products = []
if T1_ORDERS.exists():
    try:
        import pandas as pd
        orders_df = pd.read_csv(T1_ORDERS)
        available = orders_df['product'].unique().tolist()
        forecast_products = available[:3]  # use first 3 products
        print(f'Products available for forecasting: {forecast_products}')
    except Exception as e:
        print(f'Could not load orders: {e}')

if not forecast_products:
    print('No orders data found — using synthetic demonstration for 3 products.')

TREND_COLORS = {'rising': '#27ae60', 'stable': '#3498db', 'falling': '#e74c3c'}
TREND_MARKERS = {'rising': '^', 'stable': 'o', 'falling': 'v'}

fig, axes = plt.subplots(3, 1, figsize=(12, 13), sharex=False)
fig.suptitle('Seasonal Demand Forecast (AA-52)', fontsize=14, fontweight='bold', y=1.01)

synth_products = ['Tomatoes', 'Strawberries', 'Apples']
rng4 = np.random.default_rng(77)

for idx, ax in enumerate(axes):
    if forecast_products and T1_ORDERS.exists():
        product = forecast_products[idx] if idx < len(forecast_products) else synth_products[idx]
        try:
            data = forecast_demand(product, orders_path=T1_ORDERS, n_months=12)
        except Exception:
            data = []
    else:
        data = []

    if not data:
        # Synthetic data
        product = synth_products[idx]
        base = [40, 38, 45, 55, 70, 90, 100, 88, 60, 55, 48, 42]
        noise = rng4.integers(-5, 6, 12)
        import calendar
        months = [calendar.month_abbr[m] for m in range(1, 13)]
        data = [
            {
                'month_name': f'{months[i]} 2025',
                'predicted_demand': float(base[i] + noise[i] + idx * 10),
                'last_year_demand': float(base[i] + rng4.integers(-8, 9) + idx * 10),
                'trend_label': 'rising' if base[i] > base[i-1] else ('falling' if base[i] < base[i-1] else 'stable'),
            }
            for i in range(12)
        ]

    months = [d['month_name'] for d in data]
    predicted = [d['predicted_demand'] for d in data]
    last_year = [d['last_year_demand'] for d in data]
    trend_labels = [d['trend_label'] for d in data]

    # Line chart
    ax.plot(months, predicted, 'o-', color='#2c3e50', linewidth=2, label='Predicted demand', zorder=3)
    valid_ly = [(i, v) for i, v in enumerate(last_year) if v is not None]
    if valid_ly:
        ly_x = [months[i] for i, _ in valid_ly]
        ly_y = [v for _, v in valid_ly]
        ax.plot(ly_x, ly_y, 's--', color='#7f8c8d', linewidth=1.5, alpha=0.7, label='Last year actual')

    # Colour markers by trend
    for i, (m, p, tl) in enumerate(zip(months, predicted, trend_labels)):
        ax.scatter([m], [p], color=TREND_COLORS.get(tl, 'gray'),
                   marker=TREND_MARKERS.get(tl, 'o'), s=80, zorder=4)

    # Caption on peak month
    peak_idx = int(np.argmax(predicted))
    ax.annotate(
        f"Peak: {predicted[peak_idx]:.0f} units",
        xy=(months[peak_idx], predicted[peak_idx]),
        xytext=(0, 14), textcoords='offset points',
        ha='center', fontsize=8, color='#27ae60',
        arrowprops=dict(arrowstyle='->', color='#27ae60'),
    )

    ax.set_title(f'{product}', fontweight='bold')
    ax.set_ylabel('Demand (units)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    # Summary caption below chart
    peak_month = months[peak_idx]
    caption = f'High demand expected for {product} in {peak_month} based on seasonal trends.'
    ax.set_xlabel(caption, fontsize=9, style='italic')

plt.tight_layout()
plt.show()
print('\nForecast charts show predicted vs last-year demand with trend indicators (▲ rising, ▼ falling, ● stable).')

---
## AA-51: User Override Policy

When a user disagrees with a model prediction they can submit a correction via the API. This section documents the override mechanism and the retraining policy.

In [ ]:
import json

print('=== Override API (AA-51) ===')
print()
print('Endpoint: POST /interaction/<id>/override')
print('Request body: {"override_value": "<corrected prediction>"}')
print()
print('Example curl command:')
example_curl = '''curl -X POST http://localhost:5000/interaction/42/override \\
     -H "Content-Type: application/json" \\
     -d \'{ "override_value": "Grade_A" }\'\n'''
print(example_curl)

print('Example response:')
example_response = {
    'interaction_id': 42,
    'override_value': 'Grade_A',
    'status': 'overridden',
}
print(json.dumps(example_response, indent=2))

print()
print('Database fields added (AA-51):')
print('  was_overridden : BOOLEAN  — set to 1 when user submits a correction')
print('  override_value : TEXT     — the corrected prediction value')

### Retraining Policy

User overrides capture real-world corrections that the model missed. The following policy governs how overrides feed back into model improvement:

| Step | Detail |
|------|--------|
| **Collection** | Every override is stored in the `interactions` table with `was_overridden=1` and the corrected `override_value`. |
| **Batching** | Overrides are collected weekly. At the end of each week, all new overrides are exported as a labelled dataset. |
| **Threshold check** | If the weekly override rate exceeds **20%** (tracked by `override_analysis()` in `evaluate.py`) a retraining alert is raised. |
| **Retraining** | Overridden interactions are treated as additional training labels. The corrected `override_value` becomes the new ground-truth target for that input. |
| **Validation** | After retraining, the model is evaluated on a held-out set. Deployment only proceeds if the new model improves on the override examples without regressing on the original test set. |
| **Audit trail** | All overrides remain in the database with timestamps, supporting auditability and FAT review. |

This weekly feedback loop ensures the model remains aligned with operator expertise and seasonal changes in produce quality standards.